# Skripta 0 — Pivot Smoothing

**Redoslijed pokretanja:**
1. Pokreni svaki blok redom (Shift+Enter)
2. U bloku **Upload fajlova** uploaduj sva 3 CSV fajla
3. Na kraju se automatski downloaduju `pivot_smooth.csv` i `pivot_qa.csv`

**Ulazni fajlovi:**
- `pivot_3dpoly_0cm.csv` — originalna pivot linija
- `pivot_3dpoly_-1cm.csv` — donja granica koridora
- `pivot_3dpoly_+3cme.csv` — gornja granica koridora

In [ ]:
# Instalacija paketa (samo prvi put, traje ~30 sekundi)
!pip install cvxpy osqp -q

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import cvxpy as cp
from google.colab import files

print('Paketi ucitani.')

In [ ]:
# Upload fajlova — odaberi sva 3 CSV fajla odjednom
print('Uploaduj: pivot_3dpoly_0cm.csv, pivot_3dpoly_-1cm.csv, pivot_3dpoly_+3cme.csv')
uploaded = files.upload()

In [ ]:
def citaj_csv(naziv):
    """Cita CSV format: index, X, Y, Z (bez zaglavlja)"""
    df = pd.read_csv(naziv, header=None)
    return df.iloc[:, 1:4].values.astype(float)  # X, Y, Z

raw    = citaj_csv('pivot_3dpoly_0cm.csv')
lb_data = citaj_csv('pivot_3dpoly_-1cm.csv')
ub_data = citaj_csv('pivot_3dpoly_+3cme.csv')

X      = raw[:, 0]
Y      = raw[:, 1]
Z_orig = raw[:, 2]
Z_lb   = lb_data[:, 2]   # Z - 1cm (donja granica)
Z_ub   = ub_data[:, 2]   # Z + 3cm (gornja granica)
n      = len(Z_orig)

print(f'Ucitano {n} tacaka.')
print(f'Z raspon: {Z_orig.min():.3f} m — {Z_orig.max():.3f} m')
print(f'Ukupna duzina (aproks.): {n * 5:.0f} m')

In [ ]:
# Outlier detekcija — rolling median na prozoru ±10 tacaka
OUTLIER_PRAG = 0.05  # 5 cm
WINDOW = 21

Z_median = (pd.Series(Z_orig)
              .rolling(window=WINDOW, center=True, min_periods=1)
              .median()
              .values)

odstupanje = np.abs(Z_orig - Z_median)
je_outlier = odstupanje > OUTLIER_PRAG

n_outlier = int(np.sum(je_outlier))
print(f'Detektovano outlier tacaka: {n_outlier} od {n} ({100*n_outlier/n:.1f}%)')

if n_outlier > 0:
    idx_outlieri = np.where(je_outlier)[0]
    print(f'Indeksi outliera (prvih 10): {idx_outlieri[:10]}')
    print(f'Max odstupanje: {odstupanje.max()*100:.1f} cm na idx {odstupanje.argmax()}')

In [ ]:
# Korigovane granice koridora za outlier tacke
# Za outlier: koridor se bazira na interpoliranom Z iz susjednih normalnih tacaka

normalne_idx = np.where(~je_outlier)[0]
normalne_Z   = Z_orig[normalne_idx]

if len(normalne_idx) < 2:
    raise ValueError('Premalo normalnih tacaka za interpolaciju — provjeri ulazne podatke.')

interp_fn = interp1d(normalne_idx, normalne_Z, kind='linear', fill_value='extrapolate')
Z_interpol = interp_fn(np.arange(n))

# Finalne granice: outlier tacke dobijaju koridor baziran na interpoliranom Z
lb = np.where(je_outlier, Z_interpol - 0.01, Z_lb)
ub = np.where(je_outlier, Z_interpol + 0.03, Z_ub)

# Sanity check
problemi = np.sum(lb > ub + 1e-6)
if problemi > 0:
    raise ValueError(f'Greska: lb > ub na {problemi} tacaka — provjeri ulazne fajlove.')

print('Granice koridora postavljene.')
print(f'Prosjecna sirina koridora: {(ub - lb).mean()*100:.2f} cm')

In [ ]:
# QP optimizacija — minimizacija krivine uz hard constraints
# Minimizuj: suma kvadrata druge diferencije (mjera krivine)
# Uz uslov: lb_i <= Z_i <= ub_i za svaku tacku

print('Pokretanje QP optimizacije (OSQP solver)...')

Z_var = cp.Variable(n)

D2 = cp.diff(Z_var, 2)  # druga diferencija
objective   = cp.Minimize(cp.sum_squares(D2))
constraints = [Z_var >= lb, Z_var <= ub]

problem = cp.Problem(objective, constraints)
problem.solve(
    solver=cp.OSQP,
    eps_abs=1e-6,
    eps_rel=1e-6,
    max_iter=10000,
    verbose=False
)

if problem.status not in ['optimal', 'optimal_inaccurate']:
    raise ValueError(f'QP nije konvergirao. Status: {problem.status}')

Z_smooth = Z_var.value
print(f'QP status: {problem.status}')
print('Optimizacija zavrsena.')

In [ ]:
# QA analiza
delta = Z_smooth - Z_orig  # pozitivno = podignut, negativno = spusten

status = []
for i in range(n):
    if je_outlier[i]:
        status.append('OUTLIER')
    elif (Z_smooth[i] <= Z_lb[i] + 1e-4) or (Z_smooth[i] >= Z_ub[i] - 1e-4):
        status.append('KORIDOR_GRANICA')
    else:
        status.append('NORMALNO')

print('=== QA Izvjestaj ===')
print(f'  NORMALNO:          {status.count("NORMALNO")} tacaka')
print(f'  OUTLIER:           {status.count("OUTLIER")} tacaka')
print(f'  KORIDOR_GRANICA:   {status.count("KORIDOR_GRANICA")} tacaka')
print()
print('Delta Z (pomak od originala):')
print(f'  Min:  {delta.min()*100:.2f} cm')
print(f'  Max:  {delta.max()*100:.2f} cm')
print(f'  Mean: {delta.mean()*100:.2f} cm')
print(f'  Std:  {delta.std()*100:.2f} cm')

In [ ]:
# Export rezultata

# pivot_smooth.csv — ulaz za Skriptu 1
df_smooth = pd.DataFrame({
    'X': X,
    'Y': Y,
    'Z': np.round(Z_smooth, 4)
})
df_smooth.to_csv('pivot_smooth.csv', index=False, header=False)

# pivot_qa.csv — provjera rezultata
df_qa = pd.DataFrame({
    'orig_Z':   np.round(Z_orig, 4),
    'novi_Z':   np.round(Z_smooth, 4),
    'delta_cm': np.round(delta * 100, 2),
    'status':   status
})
df_qa.index.name = 'idx'
df_qa.to_csv('pivot_qa.csv', index=True)

print('Fajlovi sacuvani. Pokretanje downloada...')
files.download('pivot_smooth.csv')
files.download('pivot_qa.csv')
print('Gotovo.')